<a href="https://colab.research.google.com/github/Dan2222001/CS470-image-quality-project/blob/main/CS470_image_quality_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Imports and Google Drive Setup

To get this to work, follow these steps:
1. Open your Google Drive (Both your personal and university accounts should have the files)
2. Find the "vizwiz" folder in the "Shared with me" tab
3. Right click the vizwiz folder and press "Organize", then "Add Shortcut"
4. Select "My Drive" and add it as a shortcut

The first code block should say: "Folder exists: True" if you did it right.

In [ ]:
# Imports
import json
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms, models

from google.colab import drive

# Opening Folder From Drive
drive.mount("/content/drive")

Mounted at /content/drive


Checking which device the collab is running

In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print (f"Version: {torch.__version__}")
print (f"GPU Ready: {torch.cuda.is_available()}")
print (f"Selected device: {device}")
if device.type == "cuda":
    print (f"Device Name: {torch.cuda.get_device_name(0)}")

Version: 2.11.0+cu128
GPU Ready: True
Selected device: cuda
Device Name: NVIDIA GeForce RTX 3060


###Testing Google Drive Connection with Opening Annotations - This part is mainly just seeing what the data is like and how to display it

In [15]:
folder = Path("/content/drive/MyDrive/vizwiz/annotations")
print("Folder exists:", folder.exists())

#Printing files in the annoations folder
if folder.exists():
    for file in folder.iterdir():
        print(file.name)

#Load a dataset and print some data
with open(folder / "train.json", "r") as f:
    train_data = json.load(f)

print("Number of images:", len(train_data))
print("First annotation:", train_data[0])

In [17]:
#Printing data distributions for unrecognizability
votes = Counter(
    item["unrecognizable"]
    for item in train_data
)

for count in range(6):
    print(f"{count} votes: {votes[count]} images")

plt.bar(range(6), [votes[i] for i in range(6)])

plt.xlabel("Unrecognizable votes")
plt.ylabel("Number of images")
plt.title("Recognizability distribution")
plt.show()

In [18]:
#Printing specific image flaw distribution where values >= 3 means that the flaw is present
print(train_data[0]["flaws"])
flaws = ["BLR", "BRT", "DRK", "FRM", "OBS", "ROT"]

flaw_counts = {
    flaw: sum(item["flaws"][flaw] >= 3 for item in train_data)
    for flaw in flaws
}

plt.bar(flaw_counts.keys(), flaw_counts.values())
plt.xlabel("Quality issue")
plt.ylabel("Number of images")
plt.title("Distribution of image quality issues")
plt.show()

Class for the flaw types

In [3]:
Labels: ["BLR","BRT","DRK","FRM","OBS","OTH","NON","ROT"]

class VizWizQuality (Dataset):

    def __init__ (self, json_path, img_dir, transform):
        self.items = json.load(open(json_path))
        self.img_dir
        self.transform = Path(img_dir), transform

    def __len__ (self): return len(self.items)

    def __getitem__ (self, i): 
        it = self.items[i]
        img = Image.open (self.img_dir / it["image"]).convert("RGB")
        y = [it["flaws"][k] / 5 for k in Labels] + [it["unrecognizable"] / 5]
        return self.transform(img), torch.tensor (y, dtype=float32)
    

NameError: name 'Dataset' is not defined